In [ ]:
#Get File location
base_path = "abfss://filecontainer@synapseadlstpk.dfs.core.windows.net/Bronze_Lawer/"
#List files
print(mssparkutils.fs.ls(base_path))

StatementMeta(, , -1, SessionError, , SessionError, True)

AVAILABLE_WORKSPACE_CAPACITY_EXCEEDED: Livy session has failed. Session state: Error. Error code: AVAILABLE_WORKSPACE_CAPACITY_EXCEEDED. Your job requested 12 vcores. However, the workspace only has 0 vcores available out of quota of 12 vcores for node size family [MemoryOptimized]. Try ending the running job(s), reducing the numbers of vcores requested or increasing your vcore quota. https://learn.microsoft.com/en-us/azure/synapse-analytics/spark/apache-spark-concepts#workspace-level Source: User.

In [3]:
from pyspark.sql import *
from pyspark.sql.types import *
df = spark.read.parquet(base_path + "Customer.parquet")

display(df)

StatementMeta(Transformation, 1, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c882af1b-5164-4814-881c-df23a525f728)

In [4]:
df2 = df.select("CustomerID", "FirstName", "LastName", "EmailAddress")
display(df2)

StatementMeta(Transformation, 1, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6e599ef8-9ed7-43c6-b962-0d61f84c512e)

In [12]:
from pyspark.sql.functions import concat_ws, col
#change fullname

df2 = df.withColumn(
    "FullName",
    concat_ws(" ", col("FirstName"), col("LastName")) # " " two name ku ,middle la space kodukum
)

display(df2)

StatementMeta(Transformation, 1, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a93642b4-2564-4b81-9091-db99092b472f)

In [6]:
#check null value for all column

from pyspark.sql.functions import col, sum

df2.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df2.columns
]).show()

StatementMeta(Transformation, 1, 7, Finished, Available, Finished, False)

+----------+---------+--------+------------+
|CustomerID|FirstName|LastName|EmailAddress|
+----------+---------+--------+------------+
|         0|        0|       0|           0|
+----------+---------+--------+------------+



In [7]:
#check duplicates counts
print("Total rows:", df2.count())
print("Duplicate rows:", df2.count() - df2.dropDuplicates().count())

StatementMeta(Transformation, 1, 8, Finished, Available, Finished, False)

Total rows: 847
Duplicate rows: 0


In [10]:
#change date format
from pyspark.sql.functions import date_format, col

df2 = df.withColumn(
    "ModifiedDate",
    date_format(col("ModifiedDate"), "yyyy-MM-dd")
)

display(df2)

StatementMeta(Transformation, 1, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ec9085ab-76e9-4f45-95a9-4fe02084b806)

In [ ]:
#create FullName
from pyspark.sql.functions import concat_ws, col

df2 = df2.withColumn(
    "FullName",
    concat_ws(" ", col("FirstName"), col("LastName"))
)

In [13]:
#remove two column
df2 = df2.drop("FirstName", "LastName")

display(df2)  

StatementMeta(Transformation, 1, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1b416734-ce32-45ab-982c-48e881f661ba)

In [14]:
#change column order and save
df2 = df2.select(
    "CustomerID",
    "FullName",
    "CompanyName",
    "EmailAddress",
    "ModifiedDate"
)

display(df2)

StatementMeta(Transformation, 1, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c78035fb-11c2-4d64-89c0-55a4cb7d7a22)

In [15]:
#save as parquet
df2.write.mode("overwrite").parquet(
    base_path + "Customer_Transformed/"
)

StatementMeta(Transformation, 1, 16, Finished, Available, Finished, False)

In [16]:
#read parquet after saved properly
saved_df = spark.read.parquet(
    base_path + "Customer_Transformed/"
)

display(saved_df)

StatementMeta(Transformation, 1, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c38589f9-e0f5-445c-af66-8f5e4204beff)

In [ ]:
#Bronze to silver lawyer save as parquet
silver_path = "abfss://filecontainer@synapseadlstpk.dfs.core.windows.net/Silver_Lawer/Customer/"

df2.write.mode("overwrite").parquet(silver_path)

In [ ]:
#Read after saved to silver
silver_df = spark.read.parquet(silver_path)

display(silver_df)

In [ ]:
#Silver to Gold save as parquet
gold_path = "abfss://filecontainer@synapseadlstpk.dfs.core.windows.net/Gold_Lawer/Customer/"

df2.write.mode("overwrite").parquet(gold_path)

In [ ]:
#Read after saved to Gold
gold_df = spark.read.parquet(gold_path)

display(gold_df)